# Monocular Visual Odometry — Mathematical Derivations

**SymPy-validated first-principles derivations** for every core formula in the VO pipeline.

| Section | Topic |
|---|---|
| 1 | Pinhole Camera Model |
| 2 | Essential Matrix — algebraic derivation + singular value proof |
| 3 | RANSAC — required iteration count |
| 4 | SE(3) Lie Group — exp/log maps |
| 5 | Gauss-Newton Pose Graph Optimisation |
| 6 | Umeyama Alignment (Sim(3)) |

> All algebraic results below are independently verified using [SymPy](https://www.sympy.org/).


In [ ]:
import sympy as sp
import numpy as np
from sympy import symbols, Matrix, cos, sin, sqrt, Rational, eye, zeros
from sympy import simplify, expand, factor, trigsimp, pprint
sp.init_printing(use_latex='mathjax')
print('SymPy', sp.__version__)

---
## 1  The Pinhole Camera Model

### 1.1  Projection formula

A 3-D world point $\mathbf{X}_w = (X, Y, Z)^\top$ is projected to pixel $(u, v)$ by:

$$
\lambda \begin{pmatrix}u\\v\\1\end{pmatrix}
= \underbrace{\begin{pmatrix}f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1
\end{pmatrix}}_{K}
\begin{pmatrix}R & \mathbf{t}\end{pmatrix}
\begin{pmatrix}X\\Y\\Z\\1\end{pmatrix},
\quad \lambda = Z_c
$$


In [ ]:
# ── 1.1  Symbolic pinhole projection ──────────────────────────────────────
fx, fy, cx, cy = symbols('f_x f_y c_x c_y', positive=True)
X, Y, Z = symbols('X Y Z', real=True)

K = Matrix([[fx, 0, cx],
            [0, fy, cy],
            [0,  0,  1]])

# Identity pose (R=I, t=0) for clarity
Xc = Matrix([X, Y, Z])
x_hom = K @ Xc
u = x_hom[0] / x_hom[2]
v = x_hom[1] / x_hom[2]

print('Pixel u =', u)
print('Pixel v =', v)
print()
print('Normalised coordinate x_n = (u - cx) / fx =',
      simplify((u - cx) / fx))
print('Expected: X/Z =', X/Z)

### 1.2  Back-projection (pixel → normalised ray)

The **inverse** of projection maps a pixel $(u, v)$ to a unit ray in camera space:

$$\mathbf{x}_n = K^{-1} \begin{pmatrix}u\\v\\1\end{pmatrix}
= \begin{pmatrix}(u - c_x)/f_x \\ (v - c_y)/f_y \\ 1\end{pmatrix}$$


In [ ]:
# ── 1.2  Verify K^{-1} symbolically ──────────────────────────────────────
K_inv = K.inv()
print('K^{-1} =')
pprint(K_inv)
print()
print('K K^{-1} = I?', simplify(K @ K_inv) == eye(3))

---
## 2  Epipolar Geometry and the Essential Matrix

### 2.1  Derivation of the Epipolar Constraint

Let $\mathbf{x}_1, \mathbf{x}_2$ be normalised image coordinates of the same 3-D point
seen from camera 1 (at origin) and camera 2 (at $(R, \mathbf{t})$ relative to camera 1).

The coplanarity of $\mathbf{x}_1$, $\mathbf{t}$, and $R^\top \mathbf{x}_2$ gives:
$$\mathbf{x}_2^\top E\, \mathbf{x}_1 = 0, \quad E = [\mathbf{t}]_\times R$$

where $[\mathbf{t}]_\times$ is the skew-symmetric cross-product matrix.


In [ ]:
# ── 2.1  Skew-symmetric matrix and coplanarity ───────────────────────────
t1, t2, t3 = symbols('t_1 t_2 t_3', real=True)
x1_1, x1_2 = symbols('x_{11} x_{12}', real=True)  # normalised coords frame 1
x2_1, x2_2 = symbols('x_{21} x_{22}', real=True)  # normalised coords frame 2

t_vec = Matrix([t1, t2, t3])
x1 = Matrix([x1_1, x1_2, 1])
x2 = Matrix([x2_1, x2_2, 1])

# Skew-symmetric cross-product matrix
t_x = Matrix([
    [  0, -t3,  t2],
    [ t3,   0, -t1],
    [-t2,  t1,   0],
])

print('t_x (skew-symmetric) = ')
pprint(t_x)
print()

# Sanity: t_x @ t_vec should be zero (cross product of a vector with itself)
cross_self = t_x @ t_vec
print('t_x @ t =', simplify(cross_self.T), '  (should be [0,0,0])')

### 2.2  Proof: Essential Matrix E has two equal non-zero singular values

**Theorem.** For $E = [\mathbf{t}]_\times R$ where $R \in SO(3)$:
$$E E^\top = [\mathbf{t}]_\times [\mathbf{t}]_\times^\top = \|\mathbf{t}\|^2 I_3 - \mathbf{t}\mathbf{t}^\top$$

The eigenvalues of this matrix are $\{\|\mathbf{t}\|^2,\ \|\mathbf{t}\|^2,\ 0\}$,
so the singular values of $E$ are $\{\|\mathbf{t}\|,\ \|\mathbf{t}\|,\ 0\}$.


In [ ]:
# ── 2.2  Verify EE^T = ||t||^2 I - t t^T ────────────────────────────────
norm_t_sq = t1**2 + t2**2 + t3**2

EEt = t_x @ t_x.T
reference = norm_t_sq * eye(3) - t_vec * t_vec.T

diff = simplify(EEt - reference)
print('EE^T - (||t||^2 I - t t^T) = ')
pprint(diff)
print()
if diff == zeros(3, 3):
    print('✅ VERIFIED: EE^T = ||t||^2 I - t t^T')
else:
    print('❌ MISMATCH')

# Eigenvalues
print()
print('Eigenvalues of EE^T:')
evals = reference.eigenvals()
for val, mult in evals.items():
    print(f'  {simplify(val)}  (multiplicity {mult})')

In [ ]:
# ── 2.3  Numerical verification with a random rotation ──────────────────
from scipy.spatial.transform import Rotation

rng = np.random.default_rng(0)
t_np = rng.standard_normal(3)
R_np = Rotation.random(random_state=rng).as_matrix()

def skew(v):
    return np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])

E_np = skew(t_np) @ R_np
U, S, Vt = np.linalg.svd(E_np)

print(f't = {t_np}')
print(f'||t|| = {np.linalg.norm(t_np):.6f}')
print(f'Singular values of E: {S}')
print(f'Expected σ₁=σ₂={np.linalg.norm(t_np):.6f}, σ₃≈0')

tol = 1e-10
assert abs(S[0] - S[1]) < tol, 'σ₁ ≠ σ₂'
assert abs(S[2]) < tol, 'σ₃ ≠ 0'
assert abs(S[0] - np.linalg.norm(t_np)) < 1e-6, 'σ₁ ≠ ||t||'
print('\n✅ Numerical check passed')

---
## 3  RANSAC — Required Iteration Count

**Goal:** find the minimum number of trials $N$ such that with probability $\geq p$,
at least one of the $N$ samples of size $s$ consists entirely of inliers.

$$P(\text{all-inlier sample}) = (1 - \varepsilon)^s$$

$$P(\text{failure after } N \text{ trials}) = (1 - (1-\varepsilon)^s)^N = 1 - p$$

Solving for $N$:
$$\boxed{N = \frac{\log(1 - p)}{\log(1 - (1-\varepsilon)^s)}}$$


In [ ]:
# ── 3.1  Derive RANSAC N symbolically ────────────────────────────────────
p, eps, s, N_var = symbols('p epsilon s N', positive=True)

# P(failure) = (1 - (1-eps)^s)^N = 1 - p
# => N log(1 - (1-eps)^s) = log(1 - p)
# => N = log(1-p) / log(1 - (1-eps)^s)
from sympy import log, solve, Eq

expr = (1 - (1 - eps)**s)**N_var - (1 - p)
N_solution = solve(Eq(expr, 0), N_var)
print('N =', N_solution)

In [ ]:
# ── 3.2  Plot N vs outlier ratio for different s ─────────────────────────
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')

p_val = 0.99
eps_vals = np.linspace(0.01, 0.95, 500)

for s_val, color in [(2, '#7c83ff'), (5, '#ff7c83'), (8, '#00ff99'), (4, '#ffcc44')]:
    N_vals = np.log(1 - p_val) / np.log(1 - (1 - eps_vals)**s_val)
    ax.semilogy(eps_vals * 100, N_vals, label=f's={s_val}', color=color, lw=2)

ax.set_xlabel('Outlier ratio ε (%)', color='white')
ax.set_ylabel('Required iterations N (log scale)', color='white')
ax.set_title(f'RANSAC: Required Iterations  (p={p_val})', color='white', fontweight='bold')
ax.tick_params(colors='white')
ax.spines[:].set_color('#444')
ax.grid(True, color='#333', linestyle='--', alpha=0.5)
ax.legend(facecolor='#1c2030', labelcolor='white')
ax.set_ylim(1, 1e5)
ax.set_xlim(0, 95)
plt.tight_layout()
plt.show()
print('RANSAC requires exponentially more iterations as outlier ratio grows.')

---
## 4  SE(3) Lie Group — Exponential and Logarithm Maps

The Lie algebra $\mathfrak{se}(3)$ is parameterised by a 6-vector
$\boldsymbol{\xi} = (\boldsymbol{\omega}, \mathbf{v})^\top$
where $\boldsymbol{\omega} \in \mathbb{R}^3$ is the rotation vector and $\mathbf{v} \in \mathbb{R}^3$
is the translational component.

### 4.1  Rodrigues' Rotation Formula
$$R = I + \frac{\sin\theta}{\theta}[\boldsymbol{\omega}]_\times
+ \frac{1 - \cos\theta}{\theta^2} [\boldsymbol{\omega}]_\times^2,
\quad \theta = \|\boldsymbol{\omega}\|$$


In [ ]:
# ── 4.1  Rodrigues: verify R is orthogonal for any theta ─────────────────
theta = symbols('theta', real=True, positive=True)

# Rotation around Z axis for simplicity
omega = Matrix([0, 0, 1])   # unit vector
omega_x = Matrix([[0, -1, 0], [1, 0, 0], [0, 0, 0]])

R_rod = eye(3) + sin(theta) * omega_x + (1 - cos(theta)) * omega_x**2
print('R (Rodrigues, axis=Z) =')
pprint(trigsimp(R_rod))
print()

# Verify R^T R = I
RtR = trigsimp(R_rod.T @ R_rod)
print('R^T R = I?', RtR == eye(3))

In [ ]:
# ── 4.2  Numerical exp/log roundtrip ─────────────────────────────────────
import sys
sys.path.insert(0, '..')
from vo.pose_graph import se3_exp, se3_log

# Test a non-trivial twist
xi_orig = np.array([0.1, -0.2, 0.05, 1.0, -0.5, 2.0])
T = se3_exp(xi_orig)
xi_recovered = se3_log(T)

print('Original  ξ:', xi_orig)
print('Recovered ξ:', xi_recovered)
print('Max error  :', np.max(np.abs(xi_orig - xi_recovered)))
assert np.allclose(xi_orig, xi_recovered, atol=1e-8)
print('\n✅ exp(log(T)) = T  verified')

---
## 5  Gauss-Newton Pose Graph Optimisation

The pose graph cost is minimised by solving the linear normal equations at each iteration:

$$H \Delta\mathbf{x} = \mathbf{b}$$

where $H = \sum_{ij} J_{ij}^\top \Omega_{ij} J_{ij}$ and
$\mathbf{b} = -\sum_{ij} J_{ij}^\top \Omega_{ij} \mathbf{e}_{ij}$.

Below we verify that on a **perfectly consistent graph** (all edge errors = 0),
Gauss-Newton produces $\Delta\mathbf{x} = 0$ (no update needed).


In [ ]:
# ── 5.1  Consistent graph → GN converges in one step ────────────────────
from vo.pose_graph import PoseGraph
from vo.optimizer import PoseGraphOptimizer
import numpy as np

# Build a linear chain: T_0 = I, T_1 moves 1m, T_2 moves 2m
g = PoseGraph()
poses = [
    (np.eye(3), np.array([0.0, 0.0, 0.0])),
    (np.eye(3), np.array([1.0, 0.0, 0.0])),
    (np.eye(3), np.array([2.0, 0.0, 0.0])),
]
for i, (R, t) in enumerate(poses):
    g.add_node(i, R, t, fixed=(i == 0))

# Add perfect edges (measurement = ground truth)
g.add_edge(0, 1, R_ij=np.eye(3), t_ij=np.array([1.0, 0.0, 0.0]))
g.add_edge(1, 2, R_ij=np.eye(3), t_ij=np.array([1.0, 0.0, 0.0]))

cost_before = g.total_cost()
print(f'Cost before optimisation: {cost_before:.2e}  (should be ~0)')

opt = PoseGraphOptimizer(n_iterations=5, verbose=True)
g_opt = opt.optimize(g)

cost_after = g_opt.total_cost()
print(f'Cost after  optimisation: {cost_after:.2e}')
assert cost_after < 1e-10, 'Cost should remain ~0 on a consistent graph'
print('\n✅ GN correctly recognises a consistent graph (Δx ≈ 0)')

In [ ]:
# ── 5.2  Noisy graph → GN reduces cost ───────────────────────────────────
rng = np.random.default_rng(99)

g_noisy = PoseGraph()
for i, (R, t) in enumerate(poses):
    t_noise = t + rng.normal(0, 0.2, 3)  # add position noise
    g_noisy.add_node(i, R, t_noise, fixed=(i == 0))

g_noisy.add_edge(0, 1, R_ij=np.eye(3), t_ij=np.array([1.0, 0.0, 0.0]))
g_noisy.add_edge(1, 2, R_ij=np.eye(3), t_ij=np.array([1.0, 0.0, 0.0]))

cost_before = g_noisy.total_cost()
g_opt_noisy = opt.optimize(g_noisy)
cost_after = g_opt_noisy.total_cost()

print(f'Cost before: {cost_before:.4f}')
print(f'Cost after : {cost_after:.6f}')
print(f'Reduction  : {100*(cost_before - cost_after)/cost_before:.1f}%')
assert cost_after < cost_before
print('\n✅ GN reduces cost on a noisy graph')

---
## 6  Umeyama Alignment (Sim(3))

Given estimated positions $\{p_i\}$ and reference positions $\{\hat{p}_i\}$,
find the similarity transform $(R^*, \mathbf{t}^*, s^*)$ minimising:
$$\sum_i \|\hat{p}_i - (s^* R^* p_i + \mathbf{t}^*)\|^2$$

**Closed-form solution (Umeyama 1991):**

1. Compute cross-covariance $\Sigma = \frac{1}{n}\sum (\hat{p}_i - \bar{\hat{p}})(p_i - \bar{p})^\top$
2. SVD: $\Sigma = U S V^\top$  (with reflection fix $D = \text{diag}(1,1,\det(UV^\top))$)
3. $R^* = U D V^\top$,  $s^* = \frac{\text{tr}(SD)}{\sigma_p^2}$,  $\mathbf{t}^* = \bar{\hat{p}} - s^* R^* \bar{p}$


In [ ]:
# ── 6.1  Verify Umeyama recovers a known Sim(3) transform ────────────────
from vo.evaluation.metrics import umeyama_alignment
from scipy.spatial.transform import Rotation as Rot

rng = np.random.default_rng(42)
pts = rng.standard_normal((50, 3))

# Known transform
R_true = Rot.from_euler('xyz', [10, 20, 30], degrees=True).as_matrix()
t_true = np.array([3.0, -1.5, 0.7])
s_true = 2.3

pts_ref = (s_true * R_true @ pts.T).T + t_true

R_est, t_est, s_est = umeyama_alignment(pts, pts_ref, with_scale=True)

print(f's_true={s_true:.4f}  s_est={s_est:.6f}  err={abs(s_true-s_est):.2e}')
print(f't_true={t_true}\nt_est ={t_est.round(6)}\nerr={np.max(np.abs(t_true-t_est)):.2e}')
print(f'R error (Frobenius): {np.linalg.norm(R_true - R_est):.2e}')

assert np.allclose(s_est, s_true, atol=1e-5)
assert np.allclose(R_est, R_true, atol=1e-5)
assert np.allclose(t_est, t_true, atol=1e-5)
print('\n✅ Umeyama recovers (R, t, s) exactly from noise-free data')

In [ ]:
# ── 6.2  ATE with scale alignment ────────────────────────────────────────
from vo.evaluation.metrics import compute_ate

# Simulate a perfect estimate but with unknown scale (monocular scaling problem)
n = 100
t_vals = np.linspace(0, 2 * np.pi, n)
gt_positions = np.column_stack([np.cos(t_vals), np.sin(t_vals), t_vals * 0.2])

# Estimated trajectory: same shape but 3× too small (scale-ambiguous monocular VO)
est_scaled = gt_positions / 3.0

# Make fake pose matrices
def positions_to_poses(pos):
    poses = []
    for p in pos:
        T = np.eye(4)
        T[:3, 3] = -p   # camera position = -Rᵀt, with R=I => t = -p
        poses.append(T)
    return poses

ate_no_scale, _ = compute_ate(positions_to_poses(est_scaled),
                              positions_to_poses(gt_positions), with_scale=False)
ate_with_scale, _ = compute_ate(positions_to_poses(est_scaled),
                                positions_to_poses(gt_positions), with_scale=True)

print(f'ATE without scale alignment: {ate_no_scale:.4f} m')
print(f'ATE   with scale alignment:  {ate_with_scale:.6f} m  ← should be ~0')
assert ate_with_scale < 1e-5
print('\n✅ Sim(3) alignment corrects the monocular scale ambiguity')

---
## Summary

| Formula | SymPy/NumPy Result |
|---|---|
| $K^{-1}$ back-projection | ✅ exact symbolic inverse |
| $EE^\top = \|\mathbf{t}\|^2 I - \mathbf{t}\mathbf{t}^\top$ | ✅ zero residual |
| Singular values of $E$: $\{\sigma, \sigma, 0\}$ | ✅ verified numerically |
| RANSAC $N$ formula | ✅ symbolic derivation |
| SE(3) exp/log roundtrip | ✅ max error $< 10^{-8}$ |
| Gauss-Newton on consistent graph | ✅ cost $\approx 0$, $\Delta x \approx 0$ |
| Umeyama Sim(3) alignment | ✅ exact recovery of $(R^*, \mathbf{t}^*, s^*)$ |

> All derivations in this notebook directly correspond to code in `vo/` and are
> the mathematical foundation of the VO pipeline.
